# 🌿 Biodiversity at Scale: Master Interactive Pipeline
### Panel de Control de Experimentación en Tiempo Real
**Curso**: Aprendizaje Profundo – Práctica (Maestría en IA, UTEC Posgrado)  
**Profesor**: Dra. Aurea Soriano-Vargas

Este cuaderno interactivo centraliza la ejecución de todos los experimentos del proyecto llamando a los módulos optimizados de `src/`:
1. **Verificación de GPU dedicada** (NVIDIA RTX 5060) y precarga en memoria RAM.
2. **Monitoreo en Tiempo Real**: Barras de carga interactivas (`tqdm`) y panel de control web con **TensorBoard**.
3. **Comparación de Optimizadores SOTA**: Curvas de convergencia de error de entrenamiento y validación (SGD+M, Adam, AdamW, Muon con Newton-Schulz, Lion).
4. **Tabla Oficial de Resultados**: Métricas consolidada conforme a la Sección 14 de la rúbrica.


### 1. Inicialización y Diagnóstico de Hardware GPU


In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

import torch
from src.utils.seed import seed_everything

SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[*] Dispositivo asignado a PyTorch: {device}")
if device.type == "cuda":
    print(f"[*] GPU Dedicada: {torch.cuda.get_device_name(0)}")
    print(f"[*] VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"[*] Capacidad de Cómputo: {torch.cuda.get_device_capability(0)}")
    print(f"[*] Modo CuDNN Benchmark: {torch.backends.cudnn.benchmark}")
else:
    print("[!] Advertencia: CUDA no disponible, corriendo en CPU.")


### 2. Carga Ultrarrápida con Cache en Memoria RAM
Para eliminar los cuellos de botella de disco y permitir que la GPU NVIDIA trabaje a máxima potencia sostenida (sin esperas de I/O de CPU), precargamos la muestra en memoria RAM (`cache_in_ram=True`).


In [ ]:
from src.data.dataset import INatDataset
from src.data.transforms import get_transforms, get_batch_augmentations
from src.data.dataloader import build_dataloaders

sample_50 = ROOT_DIR.parent / "recursos" / "inat2021_sample"
data_dir = sample_50 if (sample_50.exists() and (sample_50 / "train_mini.json").exists()) else (ROOT_DIR.parent / "recursos" / "sample_inat")
print(f"[*] Directorio de datos activo: {data_dir.name}")
train_json = data_dir / "train_mini.json"
val_json = data_dir / "val.json"
train_img_dir = data_dir / "train_mini"
val_img_dir = data_dir / "val"

IMG_SIZE = 128
BATCH_SIZE = 32

tf_train = get_transforms(split="train", img_size=IMG_SIZE, aug_mode="standard")
tf_val = get_transforms(split="val", img_size=IMG_SIZE, aug_mode="none")

# Precarga en RAM (cero cuellos de botella de disco)
train_dataset = INatDataset(train_json, train_img_dir, transform=tf_train, cache_in_ram=True)
val_dataset = INatDataset(val_json, val_img_dir, transform=tf_val, category_to_label=train_dataset.category_to_label, cache_in_ram=True)

train_loader, val_loader = build_dataloaders(train_dataset, val_dataset, batch_size=BATCH_SIZE, num_workers=0, seed=SEED)

NUM_CLASSES = len(train_dataset.category_to_label)
print(f"[✓] Dataset cargado en memoria RAM:")
print(f"    - Muestras Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"    - Especies biológicas: {NUM_CLASSES}")


### 3. Lanzar Panel de Control en Tiempo Real con TensorBoard
Puedes visualizar la pérdida, precisión, F1-score y memoria VRAM en tiempo real ejecutando la siguiente celda o abriendo `http://localhost:6006` en tu navegador.


In [ ]:
# Carga de la extensión interactiva de TensorBoard en el Notebook
%load_ext tensorboard
%tensorboard --logdir ../logs/tb_runs


### 4. Comparación Experimental de Optimizadores (SGD+M vs Adam vs AdamW vs Muon vs Lion)
Ejecutamos los 5 optimizadores fijando la arquitectura convolucional y registramos sus curvas de convergencia de error.


In [ ]:
from src.models.factory import build_model
from src.training.losses import build_criterion
from src.training.optimizers import build_optimizer
from src.training.trainer import Trainer
from src.utils.visualization import plot_optimizer_error_convergence

EPOCHS_OPT = 4
criterion = build_criterion("cross_entropy")

optimizers_to_test = [
    ("SGD + Momentum", "sgd_momentum", 1e-2),
    ("Adam", "adam", 1e-3),
    ("AdamW", "adamw", 1e-3),
    ("Muon (Newton-Schulz)", "muon", 0.02),
    ("Lion (Google Brain)", "lion", 1e-4)
]

optimizer_histories = {}

for label, opt_key, lr in optimizers_to_test:
    print(f"\n=======================================================")
    print(f"  Entrenando con: {label} (lr={lr})")
    print(f"=======================================================")
    
    # Instanciamos modelo idéntico para comparación controlada
    model = build_model("resnet18", num_classes=NUM_CLASSES, pretrained=False)
    opt = build_optimizer(model, opt_type=opt_key, lr=lr)
    
    tb_path = str(ROOT_DIR / "logs" / "tb_runs" / f"opt_{opt_key}")
    trainer = Trainer(
        model=model,
        criterion=criterion,
        optimizer=opt,
        device=device,
        use_amp=True,
        tb_log_dir=tb_path
    )
    
    history, best_metrics, vram, t_time = trainer.fit(
        train_loader, val_loader, epochs=EPOCHS_OPT, verbose=True, show_pbar=True
    )
    optimizer_histories[label] = history

print("\n[✓] Todos los optimizadores evaluados con éxito.")


### 5. Gráfico de Convergencia de Error para Optimizadores
Visualizamos la tasa de error en entrenamiento ($1 - \text{Top-1}$) y la pérdida en validación para comparar directamente la velocidad de convergencia y estabilidad.


In [ ]:
plot_optimizer_error_convergence(
    optimizer_histories,
    title="Convergencia de Error y Pérdida: Comparativa de Optimizadores en iNaturalist",
    save_path=str(ROOT_DIR / "outputs" / "comparativa_convergencia_optimizadores.png")
)


### 6. Tabla Final Consolidada de Experimentos (Rúbrica Sección 14)


In [ ]:
from src.utils.tracking import ExperimentTracker

tracker = ExperimentTracker(log_dir=str(ROOT_DIR / "logs"))
print(tracker.to_markdown_table())


### 7. SOTA Elite Suite & Top 10 Modelos Campeones (224px, Swin-T, ConvNeXt, Muon Híbrido y Ensembles)
Visualización del ranking Top 10 persistido en disco (`checkpoints/top10_manifest.json`) y despliegue del gráfico comparativo 4-grid para publicación académica.


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

manifest_path = ROOT_DIR / "checkpoints" / "top10_manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        top10 = json.load(f)
    
    rows = []
    for item in top10:
        m = item["metrics"]
        hp = item.get("hyperparams", {})
        rows.append({
            "Rank": f"#{item['rank']}",
            "ID": item["exp_id"],
            "Modelo": item["model_name"],
            "Optimizador": hp.get("optimizer", "N/A"),
            "Resolución": f"{hp.get('img_size', 128)}px",
            "Top-1 Acc": f"{m.get('top1_acc', 0)*100:.2f}%",
            "Macro-F1": f"{m.get('macro_f1', 0)*100:.2f}%",
            "Top-5 Acc": f"{m.get('top5_acc', 0)*100:.2f}%",
            "Animalia Acc": f"{m.get('kingdom_animalia_acc', 0)*100:.2f}%" if "kingdom_animalia_acc" in m else "-",
            "Plantae Acc": f"{m.get('kingdom_plantae_acc', 0)*100:.2f}%" if "kingdom_plantae_acc" in m else "-"
        })
    df_top10 = pd.DataFrame(rows)
    print("🏆 TOP 10 MODELOS CAMPEONES PERSISTIDOS EN DISCO:")
    display(df_top10)

grid_img = ROOT_DIR / "outputs" / "comparativa_top_modelos_4grid.png"
if grid_img.exists():
    print(f"\n📊 Gráfica Oficial de Comparación Multimétrica (outputs/{grid_img.name}):")
    display(Image(filename=str(grid_img)))
